# Customer Behaviour Analysis for Alfido-Tech

###This project focuses on analyzing customer purchasing behavior using data-driven techniques to uncover key revenue drivers, spending patterns, and retention trends. It involves exploratory data analysis (EDA), customer segmentation (RFM), churn analysis, and cohort-based retention modeling to generate actionable business insights. The goal is to enable strategic decision-making by identifying high-value customers, optimizing product performance, and improving customer retention through targeted strategies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("/content/ecommerce_customer_data_large.csv")
df

In [ ]:
df.head()

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumns:\n", df.columns)

##Data Overview

In [ ]:
print(" Dataset Information:\n")
df.info()

print("\n Statistical Summary:\n")
display(df.describe())

print("\n Missing Values:\n")
print(df.isnull().sum())

Data Overview Insights :


The dataset contains ~250,000 records, indicating a large-scale customer transaction dataset suitable for analysis.
There are 13 columns, including customer demographics, purchase details, and churn information.
Most columns are complete, but the 'Returns' column contains missing values, which may require handling.
The dataset includes a mix of:
Numerical features (Price, Quantity, Age, Total Purchase)
Categorical features (Product Category, Payment Method, Gender)
Summary statistics show:
Average purchase amount ≈ 2700
Average customer age ≈ 44 years
No immediate anomalies in min/max values, indicating relatively clean data

##DATA-CLEANING

In [ ]:
print("🔹 Missing Values Before Cleaning:\n")
print(df.isnull().sum())

numeric_cols = df.select_dtypes(include=['int64','float64']).columns

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

duplicates = df.duplicated().sum()
print("\n🔹 Duplicate Rows:", duplicates)

df.drop_duplicates(inplace=True)

print("\n🔹 Missing Values After Cleaning:\n")
print(df.isnull().sum())

print("\n🔹 Dataset Shape After Cleaning:", df.shape)

Data Cleaning Insights :


Missing values were identified primarily in the 'Returns' column, which were handled using median imputation for numerical stability.
Categorical columns were cleaned using mode imputation, ensuring no data loss.
Duplicate records were checked and removed to maintain data integrity.
After cleaning:
The dataset is fully complete (no null values)
Data is consistent and ready for analysis
This step ensures accurate insights and reliable modeling

##Exploratory Data Analysis

In [ ]:
df_sample = df.sample(2000, random_state=42)
num_cols = ['Product Price', 'Quantity', 'Total Purchase Amount', 'Customer Age']

plt.figure(figsize=(12, 8))

for i, col in enumerate(num_cols, 1):
    plt.subplot(2, 2, i)
    plt.hist(df_sample[col], bins=30)
    plt.title(col)

plt.tight_layout()
plt.show()


cat_cols = ['Product Category', 'Payment Method', 'Gender', 'Churn']

plt.figure(figsize=(12, 8))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(2, 2, i)
    df_sample[col].value_counts().plot(kind='bar')
    plt.title(col)
    plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

Numerical Features — Deep Insights :


Most numerical features exhibit near-normal distributions, indicating stable purchasing behavior across customers.
Total Purchase Amount shows a strong central tendency (~2700), suggesting a consistent spending pattern among users.
Quantity is concentrated between 1–5, indicating:
Customers prefer low-volume, frequent purchases
Business model is likely transaction-driven rather than bulk-driven
Product Price has a wide spread → reflects a diverse product catalog (low to premium range)
Customer Age is centered around 30–55, implying:
Core customers belong to working-class / economically active segment
Strong purchasing power group

👉 Business Insight:

The platform primarily serves mid-age, consistent-spending customers with moderate purchase frequency.

Categorical Features — Behavioral Insights :


Product Category dominance indicates:
Certain categories act as revenue drivers
Others may be underperforming or niche
Payment Method trends reveal:
Strong inclination toward digital payments (PayPal / Credit Card)
Lower dependency on cash → indicates digitally mature user base
Gender distribution:
If balanced → platform is gender-neutral
If skewed → opportunity for targeted marketing campaigns

👉 Business Insight:

Customer preferences are clearly segmented by category and payment behavior, enabling targeted personalization strategies.

In [ ]:
plt.figure(figsize=(10, 6))

corr_matrix = df.corr(numeric_only=True)

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

plt.title("Correlation Heatmap of Numerical Features", fontsize=14, weight='bold')
plt.show()

Correlation Analysis — Key Business Relationships
🔹 1. Strong Positive Relationship: Quantity ↔ Total Purchase Amount
A high positive correlation exists between Quantity and Total Purchase Amount
This confirms that transaction value is volume-driven

👉 Business Interpretation:

Customers who purchase more items significantly increase revenue
Indicates strong potential for:
Bundling strategies
“Buy More, Save More” offers
🔹 2. Product Price ↔ Total Purchase Amount
Moderate to strong correlation observed between Product Price and total spend

👉 Business Interpretation:

High-priced products contribute directly to revenue growth
However, impact is less dominant than quantity

👉 Strategy:

Focus on:
Premium product promotion
Dynamic pricing strategies
🔹 3. Weak Correlation: Customer Age ↔ Spending
Customer Age shows weak correlation with Total Purchase Amount

👉 Business Interpretation:

Spending behavior is not heavily dependent on age
Indicates:
Broad appeal across age groups
No single dominant demographic

👉 Strategy:

Avoid over-segmentation by age
Focus on:
Behavioral segmentation (spending patterns, frequency)

##Customer Behavior Insights DashBoard

In [ ]:
df_sample = df.sample(5000, random_state=42)

top_categories = df_sample['Product Category'].value_counts().head(5)

df_sample['Purchase Date'] = pd.to_datetime(df_sample['Purchase Date'])
monthly_trends = df_sample.groupby(df_sample['Purchase Date'].dt.to_period('M')).size()
monthly_trends.index = monthly_trends.index.to_timestamp()

payment_counts = df_sample['Payment Method'].value_counts()

df_sample['Spending Segment'] = pd.qcut(df_sample['Total Purchase Amount'],
                                       q=3,
                                       labels=['Low', 'Medium', 'High'])
segment_counts = df_sample['Spending Segment'].value_counts()

churn_counts = df_sample['Churn'].value_counts()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Customer Behavior Analysis Dashboard", fontsize=18, weight='bold')

sns.barplot(x=top_categories.values, y=top_categories.index, ax=axes[0,0])
axes[0,0].set_title("Top Product Categories")

monthly_trends.plot(ax=axes[0,1], marker='o')
axes[0,1].set_title("Monthly Purchase Trends")
axes[0,1].grid(True)

sns.barplot(x=payment_counts.index, y=payment_counts.values, ax=axes[0,2])
axes[0,2].set_title("Payment Methods")
axes[0,2].tick_params(axis='x', rotation=30)

sns.barplot(x=segment_counts.index, y=segment_counts.values, ax=axes[1,0])
axes[1,0].set_title("Customer Segmentation")

sns.barplot(x=churn_counts.index, y=churn_counts.values, ax=axes[1,1])
axes[1,1].set_title("Churn Distribution")


axes[1,2].axis('off')

plt.tight_layout()
plt.show()

Customer Behavior Analysis — Strategic Insights (Alfido Tech)



1. Product Category Performance (Revenue Concentration)
A limited number of product categories dominate purchase volume → clear Pareto distribution (80/20 rule)
These categories act as primary revenue engines

👉 Strategic Insight:

Business is category-driven, not evenly distributed

👉 Action:

Prioritize:
Inventory optimization
Marketing spend on top categories
Re-evaluate low-performing categories:
Bundle / discount / phase out




2. Monthly Purchase Trends (Demand Intelligence)
Time-series trend shows fluctuating demand patterns
Presence of:
Peak periods → high engagement
Low periods → demand gaps

👉 Strategic Insight:

Customer activity is time-sensitive and cyclical

👉 Action:

During low-demand months:
Launch campaigns
Offer discounts
During peak periods:
Maximize revenue via:
Premium pricing
Upselling
💳 3. Payment Behavior (Digital Maturity)
Strong dominance of specific payment methods (likely digital)

👉 Strategic Insight:

Customer base is digitally mature and convenience-driven

👉 Action:

Enhance:
Payment UX
Faster checkout
Introduce:
Payment-based incentives
Cashback / partnerships
👥 4. Customer Segmentation (Spending Power)
Customers are clearly divided into:
Low, Medium, High spenders
High spenders represent:
Smaller group
Disproportionately higher revenue contribution

👉 Strategic Insight:

Business follows value-based segmentation

👉 Action:

High-value customers:
Loyalty programs
Exclusive offers
Medium:
Upsell strategies
Low:
Entry-level offers / onboarding
🔥 5. Churn Analysis (Retention Risk)
Presence of churn indicates:
Customer dissatisfaction
Weak engagement loops

👉 Strategic Insight:

Retention is a critical risk area

👉 Action:

Identify churn triggers:
Returns
Poor experience
Implement:
Retargeting campaigns
Personalized recommendations
💎 Integrated Business Insight
Revenue is driven by:
✔ High-performing categories
✔ High-value customers
Growth opportunities lie in:
✔ Increasing purchase frequency
✔ Improving retention
Risks include:
❗ Customer churn
❗ Underperforming categories

## Feature Relationships

In [ ]:
df_sample = df.sample(3000, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Feature Relationships with Total Purchase Amount", fontsize=16, weight='bold')

sns.scatterplot(
    x='Quantity',
    y='Total Purchase Amount',
    data=df_sample,
    alpha=0.5,
    ax=axes[0]
)
axes[0].set_title("Quantity vs Total Spend")
axes[0].set_xlabel("Quantity")
axes[0].set_ylabel("Total Purchase Amount")

sns.scatterplot(
    x='Product Price',
    y='Total Purchase Amount',
    data=df_sample,
    alpha=0.5,
    ax=axes[1]
)
axes[1].set_title("Price vs Total Spend")
axes[1].set_xlabel("Product Price")
axes[1].set_ylabel("")


sns.scatterplot(
    x='Customer Age',
    y='Total Purchase Amount',
    data=df_sample,
    alpha=0.5,
    ax=axes[2]
)
axes[2].set_title("Age vs Total Spend")
axes[2].set_xlabel("Customer Age")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

Feature Relationship Analysis — Revenue Drivers


🔹 1. Quantity vs Total Purchase Amount
A strong linear relationship is clearly visible
As quantity increases, total purchase amount rises proportionally

👉 Deep Insight:

Revenue is volume-driven at the transaction level
Customers tend to scale spending by increasing item count rather than switching to expensive items

👉 Business Strategy:

Introduce:
Bundle offers
Volume discounts
“Buy More, Save More” campaigns




🔹 2. Product Price vs Total Purchase Amount
Positive relationship exists but with higher dispersion
Indicates variability in how price contributes to total spend

👉 Deep Insight:

Customers purchase across different price tiers
High-priced products do not always guarantee higher total spend (depends on quantity)

👉 Business Strategy:

Use:
Cross-selling (high + low price items)
Strategic promotion of premium products
Avoid relying only on high pricing for revenue growth




🔹 3. Customer Age vs Total Purchase Amount
Scatter pattern shows no strong trend
Points are widely dispersed across all age groups

👉 Deep Insight:

Spending behavior is age-independent
Customer base is behaviorally diverse, not demographically segmented

👉 Business Strategy:

Focus on:
Behavioral segmentation (purchase frequency, spending habits)
Avoid:
Over-reliance on age-based targeting
💎 Integrated Insight (High-Level Thinking)
✔ Primary Revenue Driver: Quantity
✔ Secondary Influence: Product Price
❌ Weak Influence: Customer Age


The analysis reveals that transactional behavior—specifically purchase quantity—is the strongest driver of revenue, while demographic factors such as age have minimal influence. This highlights the importance of behavior-driven strategies for maximizing customer value.

From feature relationship analysis, I identified that revenue is primarily driven by purchase volume rather than demographics, which led to recommending behavior-based segmentation and bundling strategies.

##RFM Analysis

In [ ]:
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

reference_date = df['Purchase Date'].max()

rfm = df.groupby('Customer ID').agg({
    'Purchase Date': lambda x: (reference_date - x.max()).days,
    'Customer ID': 'count',
    'Total Purchase Amount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['Frequency'], 5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm['RFM Score'] = rfm[['R','F','M']].astype(str).agg(''.join, axis=1)

rfm.head()

##RFM Segment Visualization

In [ ]:
rfm['Segment'] = rfm['R'].astype(int) + rfm['F'].astype(int) + rfm['M'].astype(int)

plt.figure(figsize=(8,5))
rfm['Segment'].value_counts().sort_index().plot(kind='bar')

plt.title("Customer Segmentation (RFM Score)", fontsize=14, weight='bold')
plt.xlabel("Segment Score (Higher = Better)")
plt.ylabel("Number of Customers")

plt.show()

I implemented RFM segmentation to identify high-value, at-risk, and low-engagement customers, enabling targeted retention and revenue optimization strategies

##Cohort Analysis

In [ ]:
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

df['Cohort Month'] = df.groupby('Customer ID')['Purchase Date'].transform('min').dt.to_period('M')
df['Purchase Month'] = df['Purchase Date'].dt.to_period('M')

df['Cohort Index'] = (df['Purchase Month'] - df['Cohort Month']).apply(lambda x: x.n)

cohort_data = df.groupby(['Cohort Month', 'Cohort Index'])['Customer ID'].nunique().reset_index()

cohort_pivot = cohort_data.pivot(index='Cohort Month',
                                columns='Cohort Index',
                                values='Customer ID')

cohort_size = cohort_pivot.iloc[:,0]
retention = cohort_pivot.divide(cohort_size, axis=0)

plt.figure(figsize=(10,6))
sns.heatmap(retention, annot=True, fmt=".0%", cmap="Blues")

plt.title("Customer Retention (Cohort Analysis)", fontsize=14, weight='bold')
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort Month")

plt.show()

Shows how many customers come back over time
Identifies:


Strong retention cohorts

Drop-off points

In [ ]:
churn_profile = df.groupby('Churn').agg({
    'Total Purchase Amount': 'mean',
    'Quantity': 'mean',
    'Customer Age': 'mean'
})

print(churn_profile)

Compare churn vs non-churn customers


Identifies :

Lower spending customers

Less engaged users

 Helps build retention strategy


##  **Project Summary — End-to-End Business Intelligence Solution**

This project presents a comprehensive, data-driven analysis of customer behavior using the IBM HR-inspired retail dataset, transforming raw transactional data into actionable business intelligence. The workflow begins with robust data preprocessing, including data cleaning, handling missing values, feature engineering, and exploratory data analysis (EDA) to uncover underlying patterns and distributions.

Advanced analytical techniques were applied to identify key revenue drivers, revealing that purchasing behavior (quantity and price) significantly outweighs demographic influence. Product-level insights were derived using distribution and frequency analysis, highlighting high-performing categories aligned with the Pareto principle.

Customer behavior was further analyzed through statistical aggregation and visualization, identifying purchasing trends such as low-volume, high-frequency buying patterns. Payment preferences were evaluated to assess digital adoption, indicating a technologically mature customer base.

A critical component of this project is customer segmentation using RFM (Recency, Frequency, Monetary) analysis, enabling classification of customers into distinct value-based segments. This segmentation provides a foundation for targeted marketing, retention strategies, and personalized engagement.

Additionally, churn analysis was conducted to quantify customer attrition risk, while cohort analysis provided a temporal view of customer retention, emphasizing early-stage drop-off and the importance of onboarding strategies.

The project culminates in translating analytical findings into strategic business recommendations, including increasing average order value through bundling, optimizing high-performing product categories, enhancing customer retention programs, and leveraging digital payment ecosystems.

Overall, this project demonstrates the complete lifecycle of a real-world data science solution — from raw data processing to advanced analytics and strategic decision-making — delivering insights that can directly impact revenue growth, customer retention, and business optimization.

--Author

Prem N Gowda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks"

In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Customer-Behaviour-Analysisipynb.ipynb"